# vLLM Bridge — Integration Test

End-to-end validation of `boot_vllm` on a real GPU. Runs the v4 Driver protocol's vLLM backend against `meta-llama/Llama-3.2-1B`, compares per-hook activations and next-token argmax against the HF transformers backend, and exercises the affine intervention path.

**Branch:** `dev-4.x`. **Hardware:** any CUDA GPU with ≥10 GB VRAM (free Colab T4 is sufficient).

## Scope: observation + mutation

This source extends past vllm-lens's observation-only scope. Each capture hook applies an affine transform (`output = output * scale + bias`, default identity) and returns the modified tensor, so interventions propagate to downstream layers. Step 6 below is the load-bearing verification that this works end-to-end under `torch.compile` + CUDA graphs — unit tests can't reach the compiled-graph path.

## What this validates
1. `boot_vllm` returns a `RemoteBridge` end-to-end.
2. `run_with_cache` populates an `ActivationCache` from the vLLM worker via `collective_rpc`.
3. Greedy next-token argmax matches `boot_transformers` (greedy parity).
4. Per-fireable-hook relative L2 < 5e-3 vs HF (hook-fidelity gate; failures abort).
5. **Mutation smoke** (load-bearing): suppressing `embed.hook_out` zeros the cache there, shifts the argmax, and reverts cleanly on the next forward.
6. GPU release after `del bridge`.

## Setup

1. **Runtime → Change runtime type → GPU** (T4 / L4 / A100 all work).
2. **Secrets → add `HF_TOKEN`** with a token that has access to `meta-llama/Llama-3.2-1B` (gated).

The environment cell below patches `sys.stdout.fileno` because ipykernel's captured stdout doesn't expose a real file descriptor and vLLM's worker init calls `fileno()`. Without the patch, Step 2 fails with `UnsupportedOperation: fileno`.

In [ ]:
# Install vllm and TransformerLens @ dev-4.x. ~3-5 minutes.
# vllm pinned to 0.20.2 — the version the internal-API walks in
# transformer_lens/model_bridge/sources/vllm/{plugin,internals}.py were validated against.
# vLLM rearranges its internal class paths every 4-6 weeks; re-validate before bumping.
%pip install -q "vllm==0.20.2"
%pip install -q git+https://github.com/TransformerLensOrg/TransformerLens.git@dev-4.x

In [ ]:
import gc
import os
import sys

import torch

# HF_TOKEN comes from Colab secrets. Falls back to env var for non-Colab runs.
try:
    from google.colab import userdata
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN"))
except (ImportError, Exception):
    pass
assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in Colab Secrets (gear icon, left sidebar)."

# Colab/Jupyter compatibility: ipykernel's stdout doesn't expose a fileno();
# vLLM's worker init calls sys.stdout.fileno() during parallel-state setup
# and crashes with UnsupportedOperation: fileno. Patch fileno to return the
# underlying process FDs (1, 2) — Colab writes back to those anyway.
if "ipykernel" in sys.modules:
    sys.stdout.fileno = lambda: 1  # type: ignore[method-assign]
    sys.stderr.fileno = lambda: 2  # type: ignore[method-assign]

MODEL = "meta-llama/Llama-3.2-1B"
PROMPT = "The quick brown fox jumps over the"
DTYPE = torch.float16
torch.manual_seed(0)

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only — abort'}")
assert torch.cuda.is_available(), "GPU runtime required."

## Step 1 — HF reference

Boot the transformers backend first, capture activations and argmax, then drop it so vLLM has the GPU to itself.

In [ ]:
from transformers import AutoConfig

from transformer_lens.model_bridge.sources.transformers import boot as boot_transformers
from transformer_lens.model_bridge.sources.vllm.overlays import get_overlay

# Compute the expected vLLM-fireable hook set from the overlay so HF captures
# only what we'll compare. Avoids hundreds of MB of CPU clones for hooks vLLM
# doesn't expose.
_hf_preview = AutoConfig.from_pretrained(MODEL, token=os.environ["HF_TOKEN"])
_overlay = get_overlay(_hf_preview.architectures[0])
EXPECTED_HOOKS = set(_overlay.capture_specs(_hf_preview).keys())
print(f"Expecting {len(EXPECTED_HOOKS)} fireable hook(s) per vLLM overlay.")

bridge_hf = boot_transformers(MODEL, dtype=DTYPE).to("cuda")
tokens = bridge_hf.to_tokens(PROMPT)
logits_hf, cache_hf = bridge_hf.run_with_cache(
    tokens, names_filter=lambda name: name in EXPECTED_HOOKS
)
argmax_hf = int(logits_hf[0, -1].argmax().item())

cache_hf_cpu = {name: t.detach().cpu().clone() for name, t in cache_hf.cache_dict.items()}
next_token_hf = bridge_hf.tokenizer.decode([argmax_hf])
print(f"HF argmax token id: {argmax_hf} → {next_token_hf!r}")
print(f"HF cache: {len(cache_hf_cpu)} entries (filtered to overlay's fireable set)")

del bridge_hf, logits_hf, cache_hf
gc.collect(); torch.cuda.empty_cache()
print(f"GPU memory after HF release: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Step 2 — Boot vLLM bridge

`boot_vllm` constructs the LLM, monkey-patches `Worker.load_model` pre-compile to install capture hooks, then wraps it in a `RemoteBridge`.

In [ ]:
from transformer_lens.model_bridge.remote_bridge import RemoteBridge

bridge = RemoteBridge.boot_vllm(MODEL, dtype=DTYPE, gpu_memory_utilization=0.5)
assert isinstance(bridge, RemoteBridge), f"Expected RemoteBridge, got {type(bridge).__name__}"
print(f"Architecture: {bridge.cfg.architecture}")
print(f"Fireable hooks: {len(bridge._driver.supported_hook_points)}")
print(f"Non-fireable hooks (fused kernels): {len(bridge._driver.non_fireable_hook_points)}")

## Step 3 — Capture pipeline

Run a single forward, populate the cache via `collective_rpc → tl_read_captures`, and sanity-check shapes.

In [ ]:
tokens = bridge.to_tokens(PROMPT)
logits_vllm, cache_vllm = bridge.run_with_cache(tokens)
argmax_vllm = int(logits_vllm[0, -1].argmax().item())
next_token_vllm = bridge.tokenizer.decode([argmax_vllm])

print(f"vLLM argmax token id: {argmax_vllm} → {next_token_vllm!r}")
print(f"vLLM cache entries: {len(cache_vllm.cache_dict)}")

for name in sorted(bridge._driver.supported_hook_points)[:5]:
    t = cache_vllm[name]
    finite_pct = 100 * torch.isfinite(t).float().mean().item()
    print(f"  {name}: shape={tuple(t.shape)}, dtype={t.dtype}, finite={finite_pct:.1f}%")

## Step 4 — Greedy parity 

vLLM and HF must produce the same next-token argmax on the same prompt.

In [ ]:
parity = argmax_vllm == argmax_hf
status = "✅ PASS" if parity else "❌ FAIL"
print(f"{status}: HF→{argmax_hf} ({next_token_hf!r}) vs vLLM→{argmax_vllm} ({next_token_vllm!r})")
assert parity, "Greedy parity failed — kernel divergence or overlay misconfiguration."

## Step 5 — Per-hook L2 (acceptance gate)

Target is relative L2 < 5e-3 in fp16. Known caveat: an earlier spike found ~31× divergence on layer-0 attention output between HF and vLLM (kernel-fusion boundary mismatch); failures here likely indicate an overlay-side hook is wrong or the architecture needs per-hook tolerance overrides. This is the hook-fidelity gate — failing hooks abort.

In [ ]:
TARGET_REL_L2 = 5e-3
rows = []
for name in sorted(bridge._driver.supported_hook_points):
    if name not in cache_hf_cpu:
        rows.append((name, None, None, "missing in HF cache"))
        continue
    t_vllm = cache_vllm[name].detach().cpu().float()
    t_hf = cache_hf_cpu[name].float()
    if t_vllm.shape != t_hf.shape:
        rows.append((name, None, None, f"shape {tuple(t_vllm.shape)} vs HF {tuple(t_hf.shape)}"))
        continue
    diff = (t_vllm - t_hf).norm().item()
    base = t_hf.norm().item() or 1.0
    rel = diff / base
    mark = "✅" if rel < TARGET_REL_L2 else "❌"
    rows.append((name, rel, mark, ""))

print(f"{'hook':<40} {'rel L2':>10}  status  note")
print("-" * 80)
for name, rel, mark, note in rows:
    rel_str = f"{rel:.3e}" if isinstance(rel, float) else "—"
    print(f"{name:<40} {rel_str:>10}  {mark or '—':<6}  {note}")

failed = [(name, rel) for name, rel, _, _ in rows if rel is not None and rel >= TARGET_REL_L2]
assert not failed, f"Hooks exceeded fp16 drift (target {TARGET_REL_L2}): {failed}"
missing = [(name, note) for name, rel, _, note in rows if rel is None]
assert not missing, f"Hooks missing or shape-mismatched vs HF: {missing}"


## Step 6 — Intervention smoke (load-bearing)

**This cell is the load-bearing verification for the vLLM source's mutation claim.** Unit tests only exercise the dispatch protocol (mocked LLM); they cannot prove that the affine math (`output = output * scale + bias`) traces correctly through `torch.compile` and propagates to downstream layers under CUDA-graph replay. That guarantee depends on this cell passing.

The test: zero out `embed.hook_out` via `{"op": "suppress"}`. If the mutation path works, (a) the cache shows zeros there, (b) the next-token argmax shifts vs the clean run, and (c) the immediately-following clean forward reverts (no sticky state).

In [ ]:
# Suppress (zero) the embedding output. Forward should behave very differently.
logits_suppressed, cache_suppressed = bridge.run_with_cache(
    tokens,
    intervene={"embed.hook_out": {"op": "suppress"}},
)
argmax_suppressed = int(logits_suppressed[0, -1].argmax().item())

# Embed cache should be all zeros after suppress.
embed_norm = cache_suppressed["embed.hook_out"].abs().max().item()
print(f"Embed |max| after suppress: {embed_norm:.6f} (should be 0.0)")
assert embed_norm == 0.0, "Suppress did not zero embed.hook_out — intervention path broken."

# Argmax should differ from the clean run.
argmax_shifted = argmax_suppressed != argmax_vllm
print(f"Clean argmax: {argmax_vllm}  Suppressed argmax: {argmax_suppressed}  Shifted: {argmax_shifted}")

# Verify the next forward (no intervene) reverts — interventions are not sticky.
logits_revert, _ = bridge.run_with_cache(tokens)
argmax_revert = int(logits_revert[0, -1].argmax().item())
print(f"Revert argmax: {argmax_revert}  matches clean: {argmax_revert == argmax_vllm}")
assert argmax_revert == argmax_vllm, "Intervention persisted across calls — reset path broken."

## Step 7 — Lifetime

GPU memory must drop back near zero after `del bridge` + `gc.collect()` + `empty_cache()`. Long-running notebooks must not leak hooks or buffers.

In [ ]:
before = torch.cuda.memory_allocated() / 1e9
bridge.close()
del bridge, logits_vllm, cache_vllm, logits_suppressed, cache_suppressed, logits_revert
gc.collect(); torch.cuda.empty_cache()
after = torch.cuda.memory_allocated() / 1e9
print(f"GPU memory  before del: {before:.2f} GB  after del: {after:.2f} GB")
# Soft target: < 1 GB residual (some libs hold caches; the precise threshold depends on CUDA driver).
if after > 1.0:
    print(f"⚠ {after:.2f} GB residual — review for retained references.")
else:
    print("✅ GPU released cleanly.")

## Summary

If all asserts above passed, the v4 Driver-protocol vLLM backend is sound on this architecture:

- `boot_vllm` returns `RemoteBridge` end-to-end.
- `collective_rpc → tl_read_captures` populates the cache.
- Greedy argmax matches HF (greedy parity).
- Per-hook L2 < 5e-3 vs HF across every fireable hook (hook-fidelity gate).
- Affine interventions (suppress / scale / add / set) apply per-forward and reset cleanly.
- GPU lifetime is well-behaved.

Next: extend to other architectures (Qwen / Mistral / Gemma) via the `DecoderOnlyOverlay`, or add `clamp` to the intervention vocabulary.